<a href="https://colab.research.google.com/github/Nadiax94/week3-labs/blob/main/W3D3_engine_swap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Predict Card

1. At concurrency 8, vLLM throughput will be about 1.4x Monday's static-batch-8 number.

2. Monday static batching scaled from batch 1 to batch 8 by 2.87x.

3. I predict vLLM will scale from concurrency 1 to 8 by 4.0x.

4. vLLM's scaling multiple should be larger than static batching's.

5. I expect it to be roughly 1.4x larger.

In [ ]:
# RECOVERY CELL
import os, sys, time, signal, subprocess, urllib.request, urllib.error

RECOVERY_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
RECOVERY_PORT = 8000
RECOVERY_LOG = "/content/server.log"

_R_TRANSFORMERS = "4.46.*"
_R_ACCELERATE = "1.1.*"
_R_NEED_AWQ = False
_R_VLLM = "0.6.*"
_R_HTTPX = "0.27.*"
_R_OPENAI = "1.54.*"

RECOVERY_ARGS = {
    "--model": RECOVERY_MODEL,
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": str(RECOVERY_PORT),
}

# 1) kill any leftover server
subprocess.run(
    ["pkill", "-f", "vllm.entrypoints.openai.api_server"],
    check=False
)
time.sleep(2)

# 2) reinstall pins
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        f"vllm=={_R_VLLM}",
        f"transformers=={_R_TRANSFORMERS}",
        f"accelerate=={_R_ACCELERATE}",
        f"httpx=={_R_HTTPX}",
        f"openai=={_R_OPENAI}",
    ] + (["autoawq==0.2.9"] if _R_NEED_AWQ else []),
    check=True
)

print("pins reinstalled")

# 3) relaunch server
_r_cmd = [
    sys.executable,
    "-m",
    "vllm.entrypoints.openai.api_server"
]

for k, v in RECOVERY_ARGS.items():
    _r_cmd += [k] if v is None else [k, str(v)]

_r_logf = open(RECOVERY_LOG, "wb")

server = subprocess.Popen(
    _r_cmd,
    stdout=_r_logf,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

print(
    f"relaunched server pid {server.pid}, "
    f"logging to {RECOVERY_LOG}"
)

# 4) health poll
_deadline = time.time() + 300

while time.time() < _deadline:
    try:
        with urllib.request.urlopen(
            f"http://localhost:{RECOVERY_PORT}/v1/models",
            timeout=5
        ) as r:
            if r.status == 200:
                print(
                    "RECOVERED: server healthy. "
                    "continue from your last step."
                )
                break
    except (
        urllib.error.URLError,
        ConnectionError,
        OSError
    ):
        pass

    time.sleep(3)

else:
    print("recovery timed out. last 30 log lines:")

    try:
        with open(
            RECOVERY_LOG,
            errors="replace"
        ) as fh:
            print("".join(fh.readlines()[-30:]))

    except FileNotFoundError:
        print("(no log file)")

In [6]:
import sys
import subprocess

p = subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--dry-run",
        "vllm==0.6.*",
        "transformers==4.46.*",
        "accelerate==1.1.*",
        "httpx==0.27.*",
        "openai==1.54.*",
    ],
    capture_output=True,
    text=True
)

print("RETURN CODE:", p.returncode)

print("\n===== STDOUT =====")
print(p.stdout)

print("\n===== STDERR =====")
print(p.stderr)

RETURN CODE: 1

===== STDOUT =====
  Using cached vllm-0.6.6.post1-cp38-abi3-manylinux1_x86_64.whl.metadata (11 kB)
  Using cached transformers-4.46.3-py3-none-any.whl.metadata (44 kB)
  Using cached accelerate-1.1.1-py3-none-any.whl.metadata (19 kB)
  Using cached httpx-0.27.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached openai-1.54.5-py3-none-any.whl.metadata (24 kB)
  Using cached numpy-1.26.4.tar.gz (15.8 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached blake3-1.0.9-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.9 kB)
  Using cached prometheus_fastapi_ins

In [1]:
!nvidia-smi

Tue Sep  1 08:03:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import sys
print(sys.version)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [3]:
import subprocess, sys

VLLM_PIN = "0.6.*"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"
HTTPX_PIN = "0.27.*"
OPENAI_PIN = "1.54.*"

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    f"vllm=={VLLM_PIN}",
    f"transformers=={TRANSFORMERS_PIN}",
    f"accelerate=={ACCELERATE_PIN}",
    f"httpx=={HTTPX_PIN}",
    f"openai=={OPENAI_PIN}",
], check=True)

print("serving pins installed")

serving pins installed


In [4]:
import os, signal, subprocess, sys

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
PORT = 8000
SERVER_LOG = "/content/server.log"

SERVER_ARGS = {
    "--model": MODEL,
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": str(PORT),
}

cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server"]

for k, v in SERVER_ARGS.items():
    cmd += [k, str(v)]

logf = open(SERVER_LOG, "wb")

server = subprocess.Popen(
    cmd,
    stdout=logf,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

print("server pid:", server.pid)
print("log:", SERVER_LOG)

server pid: 2223
log: /content/server.log


In [5]:
import time
import urllib.request
import urllib.error

url = "http://localhost:8000/v1/models"

for i in range(100):
    try:
        with urllib.request.urlopen(url, timeout=5) as r:
            if r.status == 200:
                print("SERVER HEALTHY:", url, "-> 200")
                break
    except:
        pass

    time.sleep(3)

else:
    print("SERVER NOT READY")
    print("\nLAST LOG LINES:\n")
    with open("/content/server.log", "r", errors="replace") as f:
        print("".join(f.readlines()[-30:]))

SERVER HEALTHY: http://localhost:8000/v1/models -> 200


In [6]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="not-needed"
)

r = client.chat.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    messages=[
        {
            "role": "user",
            "content": "In one sentence, what is a GPU?"
        }
    ],
)

print(r.choices[0].message.content)

A GPU, or Graphics Processing Unit, is a specialized processor designed to accelerate computations involved in rendering graphics and video content on electronic devices.


In [8]:
from google.colab import files
uploaded = files.upload()

import json

baseline = json.load(open("baselines.json"))

print("baseline batch tokens/s:", baseline["batch"])

Saving baselines.json to baselines.json
baseline batch tokens/s: {'1': 36.0, '4': 52.4, '8': 103.4}


In [9]:
from google.colab import files
import os

if not os.path.exists("ab_client.py"):
    print("اختر ملف ab_client.py")
    files.upload()

print("ab_client.py ready")

ab_client.py ready


In [10]:
%run ab_client.py

print("ab_client loaded")
print("Number of prompts:", len(FIXED_PROMPTS))

ab_client loaded
Number of prompts: 8


In [11]:
prompts = FIXED_PROMPTS

vllm_measured = await run_sweep(
    base_url="http://localhost:8000/v1",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    prompts=prompts,
    concurrencies=[1, 4, 8],
)

for level in vllm_measured:
    print(level)

level: {'concurrency': 1, 'requests': 24, 'tokens_per_s': 59.1, 'wall_s': 23.487}
level: {'concurrency': 4, 'requests': 24, 'tokens_per_s': 170.0, 'wall_s': 8.17}
level: {'concurrency': 8, 'requests': 24, 'tokens_per_s': 222.5, 'wall_s': 6.244}
{'concurrency': 1, 'requests': 24, 'tokens_per_s': 59.1, 'wall_s': 23.487}
{'concurrency': 4, 'requests': 24, 'tokens_per_s': 170.0, 'wall_s': 8.17}
{'concurrency': 8, 'requests': 24, 'tokens_per_s': 222.5, 'wall_s': 6.244}


In [12]:
import json

def tokps_at(level_list, c):
    return next(
        x["tokens_per_s"]
        for x in level_list
        if x["concurrency"] == c
    )

vllm_by_c = {
    x["concurrency"]: x["tokens_per_s"]
    for x in vllm_measured
}

base_by_c = {
    int(k): v
    for k, v in baseline["batch"].items()
}

speedup = {
    c: round(vllm_by_c[c] / base_by_c[c], 2)
    for c in vllm_by_c
    if c in base_by_c
}

report = {
    "baseline": base_by_c,
    "vllm": vllm_by_c,
    "speedup_by_concurrency": speedup,
    "predicted_speedup": 1.4
}

with open("ab_report.json", "w") as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))

{
  "baseline": {
    "1": 36.0,
    "4": 52.4,
    "8": 103.4
  },
  "vllm": {
    "1": 59.1,
    "4": 170.0,
    "8": 222.5
  },
  "speedup_by_concurrency": {
    "1": 1.64,
    "4": 3.24,
    "8": 2.15
  },
  "predicted_speedup": 1.4
}


In [13]:
static_scaling = base_by_c[8] / base_by_c[1]
vllm_scaling   = vllm_by_c[8] / vllm_by_c[1]

print(
    f"static batching scales {static_scaling:.2f}x, "
    f"vLLM scales {vllm_scaling:.2f}x"
)

print(
    f"continuous batching is worth "
    f"{vllm_scaling / static_scaling:.2f}x of scaling"
)

static batching scales 2.87x, vLLM scales 3.76x
continuous batching is worth 1.31x of scaling


In [14]:
c8 = next(
    x for x in vllm_measured
    if x["concurrency"] == 8
)

print("Concurrency 8 throughput:")
print(c8["tokens_per_s"], "tokens/s")

Concurrency 8 throughput:
222.5 tokens/s


In [15]:
scaling_gain = vllm_scaling / static_scaling
speedup8 = vllm_by_c[8] / base_by_c[8]

reflection = f"""
Reflection

Static batching scaled from concurrency 1 to 8 by {static_scaling:.2f}x,
while vLLM scaled by {vllm_scaling:.2f}x.

Continuous batching therefore provided about {scaling_gain:.2f}x better scaling.

At concurrency 8, vLLM was {speedup8:.2f}x faster than Monday's static batch-8 baseline.

The scaling multiple is the better number to quote to a capacity planner
because the per-concurrency speedup is a ratio of two throughput curves and
can vary with queue depth and measurement noise.

vLLM scales better because continuous batching can remove a finished sequence
and immediately admit another waiting request. Static batching cannot reuse
finished slots as efficiently when requests have different output lengths.
"""

print(reflection)


Reflection

Static batching scaled from concurrency 1 to 8 by 2.87x,
while vLLM scaled by 3.76x.

Continuous batching therefore provided about 1.31x better scaling.

At concurrency 8, vLLM was 2.15x faster than Monday's static batch-8 baseline.

The scaling multiple is the better number to quote to a capacity planner
because the per-concurrency speedup is a ratio of two throughput curves and
can vary with queue depth and measurement noise.

vLLM scales better because continuous batching can remove a finished sequence
and immediately admit another waiting request. Static batching cannot reuse
finished slots as efficiently when requests have different output lengths.



Reflection

Static batching scaled from concurrency 1 to 8 by 2.87x,
while vLLM scaled by 3.76x.

Continuous batching therefore provided about 1.31x better scaling.

At concurrency 8, vLLM was 2.15x faster than Monday's static batch-8 baseline.

The scaling multiple is the better number to quote to a capacity planner
because the per-concurrency speedup is a ratio of two throughput curves and
can vary with queue depth and measurement noise.

vLLM scales better because continuous batching can remove a finished sequence
and immediately admit another waiting request. Static batching cannot reuse
finished slots as efficiently when requests have different output lengths.

In [16]:
import os
import signal
import time
import urllib.request
import urllib.error

def shutdown_server(proc=None, port=8000):
    try:
        proc = server if proc is None else proc
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        print(f"sent SIGTERM to process group of pid {proc.pid}")
    except (ProcessLookupError, NameError):
        print("no server process to kill")

    time.sleep(3)

    try:
        with urllib.request.urlopen(
            f"http://localhost:{port}/v1/models",
            timeout=2
        ):
            print(f"WARNING: port {port} still answering")
    except (urllib.error.URLError, ConnectionError, OSError):
        print(f"port {port} is free")

shutdown_server()

sent SIGTERM to process group of pid 2223
port 8000 is free


In [17]:
%run verify_cell.py

baseline batch-8: 103.4, vllm concurrency-8: 222.5
speedup at 8: 2.15x
GREEN CHECK: PASS
